# Squat-form classifier — tuning experiment

Improves on the RF baseline (acc 0.884 / macroF1 0.883). The 6 classes are balanced
(~1581 each), so `asymmetric_squat` (recall 0.63) is a **separability** problem, not
imbalance. This notebook compares several models on the SAME split (seed 42), reports
per-class recall + feature importances, and saves the best model to
`runs/squat_form_classifier_rf_v2/`.

Run top to bottom. Complete the Kaggle login form before the experiment cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/health_training'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
print(DRIVE_ROOT)

In [ ]:
# Clone the model-training branch. %cd /content first so re-running is safe.
%cd /content
!rm -rf /content/health_trainer
!git clone --branch model-training --single-branch https://github.com/kimgt0128/health-trainer.git /content/health_trainer
%cd /content/health_trainer
!pip install -q "kagglehub[pandas-datasets]"

In [ ]:
# Kaggle auth — enter username + API token, wait for green confirmation, then continue.
import kagglehub
kagglehub.login()

In [ ]:
# Load data once, single 80/20 split (seed 42, same as the baseline for comparability).
import sys; sys.path.insert(0, '/content/health_trainer/ml/src')
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import recall_score
from healthtrainer_ml.squat_pose_dataset import (
    LABELS, load_squat_dataframe, split_features_labels, feature_config)
from healthtrainer_ml.metrics import accuracy, macro_f1, confusion_matrix

X, y = split_features_labels(load_squat_dataframe())
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
ASYM = 5  # asymmetric_squat label index

def run(name, model):
    model.fit(Xtr, ytr)
    p = model.predict(Xte)
    acc = accuracy(yte.tolist(), p.tolist())
    f1 = macro_f1(yte.tolist(), p.tolist(), len(LABELS))
    asym = recall_score(yte, p, average=None)[ASYM]
    print(f"{name:30s}  acc={acc:.4f}  macroF1={f1:.4f}  asym_recall={asym:.3f}")
    return dict(name=name, model=model, acc=acc, f1=f1, asym=float(asym))

print(f"train={len(ytr)}  test={len(yte)}\n")
configs = [
    ("RF 100 (baseline)",          RandomForestClassifier(n_estimators=100, random_state=42)),
    ("RF 300",                     RandomForestClassifier(n_estimators=300, random_state=42)),
    ("RF 300 balanced_subsample",  RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample', random_state=42)),
    ("RF 400 leaf2 sqrt",          RandomForestClassifier(n_estimators=400, min_samples_leaf=2, max_features='sqrt', random_state=42)),
    ("HistGradientBoosting",       HistGradientBoostingClassifier(random_state=42)),
]
results = [run(n, m) for n, m in configs]

In [ ]:
# Pick the best by macro-F1, save its 4 artifacts to runs/squat_form_classifier_rf_v2/.
import os, json, joblib
best = max(results, key=lambda r: r['f1'])
print("BEST by macroF1:", best['name'],
      f"(acc={best['acc']:.4f}, macroF1={best['f1']:.4f}, asym_recall={best['asym']:.3f})\n")

RUN = f"{RUNS_DIR}/squat_form_classifier_rf_v2"
os.makedirs(RUN, exist_ok=True)
joblib.dump(best['model'], f"{RUN}/squat_form_classifier.joblib", compress=3)
p = best['model'].predict(Xte)
metrics = {"squat_form_classifier": {
    "model": best['name'], "accuracy": best['acc'], "macro_f1": best['f1'],
    "asymmetric_recall": best['asym'],
    "confusion_matrix": confusion_matrix(yte.tolist(), p.tolist(), len(LABELS)),
    "n_train": int(len(ytr)), "n_test": int(len(yte))}}
json.dump(metrics, open(f"{RUN}/metrics_summary.json", "w"), indent=2)
json.dump({str(k): v for k, v in LABELS.items()}, open(f"{RUN}/labels_squat_form.json", "w"), indent=2)
json.dump(feature_config(), open(f"{RUN}/feature_config.json", "w"), indent=2)
print("saved ->", RUN)

In [ ]:
# Feature importances (which signals drive the prediction) — from a 300-tree RF.
rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(Xtr, ytr)
print("feature importances (desc):")
for nm, imp in sorted(zip(feature_config()['features'], rf.feature_importances_), key=lambda z: -z[1]):
    print(f"  {nm:20s} {imp:.3f}")